# Layers & the Forward Pass

This notebook accompanies the **ML Viz** lesson on layers and the forward pass.
We'll write the forward pass in matrix form with NumPy, reproduce the lesson's
2→2→1 worked example exactly, and visualize what hidden width does to the
functions a network can express.

**Companion lesson:** https://ml-viz.vercel.app/courses/neural-networks/03-layers-and-forward-pass

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The layer equation

A layer of $m$ neurons reading $n$ inputs is one matrix multiplication:

$$\mathbf{h} = f\left(W\mathbf{x} + \mathbf{b}\right), \qquad W \in \mathbb{R}^{m \times n},\; \mathbf{b} \in \mathbb{R}^m$$

- Each **row** of $W$ is one neuron's weight vector (rows = outputs, columns = inputs)
- The activation $f$ is applied **element-wise**
- A deep network is just this rule applied repeatedly, layer after layer

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# The 2 -> 2 -> 1 network from the lesson:
W1 = np.array([[2.0, -1.0],     # hidden neuron 1 (row 1)
               [0.5,  1.0]])    # hidden neuron 2 (row 2)
b1 = np.array([0.5, -3.0])
W2 = np.array([[2.0, 3.0]])     # output neuron weights the 2 hidden activations
b2 = np.array([1.0])

x = np.array([1.0, 2.0])

# Step 1 - hidden pre-activations (each row of W1 dots with x)
z1 = W1 @ x + b1
print('z1 =', z1)                       # [ 0.5 -0.5]

# Step 2 - ReLU element-wise: neuron 2 is clamped to 0 (inactive)
h1 = np.maximum(0, z1)
print('h1 =', h1)                       # [0.5 0. ]

# Step 3 - output layer + sigmoid
z2 = W2 @ h1 + b2
y = sigmoid(z2)
print('z2 =', z2)                       # [2.]
print('y  =', y.round(4))               # [0.8808] -> 'class 1' with p ~ 0.88

## A reusable forward pass

The same three lines generalize to any number of layers — and to whole
**batches**: stack $B$ examples as the rows of $X \in \mathbb{R}^{B \times n}$ and
compute $H = f(XW^\top + \mathbf{b})$. Nothing else changes.

In [ ]:
def init_network(sizes, seed=42):
    """Deterministic weights for a layer-size sequence, e.g. [2, 3, 1]."""
    rng = np.random.RandomState(seed)
    return [(rng.randn(m, n) * np.sqrt(2 / n), np.zeros(m))
            for n, m in zip(sizes[:-1], sizes[1:])]

def forward(X, params, act=np.tanh):
    """Forward pass for a batch X of shape (B, n). Output layer is linear."""
    H = X
    for i, (W, b) in enumerate(params):
        Z = H @ W.T + b
        H = Z if i == len(params) - 1 else act(Z)   # no activation on output
    return H

params = init_network([2, 3, 3, 1])
X_batch = np.array([[1.0, 2.0],
                    [0.0, 0.0],
                    [-1.5, 0.5]])

print('layer shapes:', [W.shape for W, _ in params])   # [(3,2), (3,3), (1,3)]
print('batch output:\n', forward(X_batch, params).round(4))
n_params = sum(W.size + b.size for W, b in params)
print('total parameters:', n_params)

## What hidden width does

Each hidden ReLU neuron contributes one "fold line" to the function the
network expresses. Few neurons → a coarse, piecewise-linear surface; many
neurons → a richly curved one. Let's draw the output surface of random
2 → h → 1 ReLU networks for increasing width $h$.

In [ ]:
relu = lambda z: np.maximum(0, z)

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Output surface of a random 2 → h → 1 ReLU network', color='white', y=1.02)

for ax, h in zip(axes, [2, 8, 64]):
    params = init_network([2, h, 1], seed=7)
    Z = forward(grid, params, act=relu).reshape(xx.shape)
    im = ax.contourf(xx, yy, Z, levels=24, cmap='RdBu_r')
    ax.set_title(f'h = {h}  ({4 * h + 1} params)', color='white', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## Depth composes features

Width adds folds *within* one layer; depth lets later layers fold the
*already-folded* surface again. With the same parameter budget, a deeper
network produces visibly more intricate structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Same idea, deeper: random tanh networks', color='white', y=1.02)

architectures = [[2, 8, 1], [2, 8, 8, 1], [2, 8, 8, 8, 1]]
for ax, sizes in zip(axes, architectures):
    params = init_network(sizes, seed=3)
    Z = forward(grid, params, act=np.tanh).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=24, cmap='RdBu_r')
    n = sum(W.size + b.size for W, b in params)
    ax.set_title(' → '.join(map(str, sizes)) + f'  ({n} params)', color='white', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## Key takeaways

- A layer is one matrix multiplication plus an element-wise activation: $\mathbf{h} = f(W\mathbf{x} + \mathbf{b})$, with $mn + m$ parameters for $n \to m$.
- The forward pass alternates matrix products and activations; batches come for free by stacking inputs as rows.
- ReLU clamps negative pre-activations to zero — inactive neurons contribute nothing for that input.
- **Width** adds capacity within a layer; **depth** composes features, which is far more parameter-efficient.
- Without the non-linearity between layers, any stack of matrices collapses to a single linear map.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — A dense layer

One layer is one affine map plus a nonlinearity, applied to a whole **batch** at once:

$$H = \sigma(XW + \mathbf{b}), \qquad X \in \mathbb{R}^{n \times d_{in}}, \; W \in \mathbb{R}^{d_{in} \times d_{out}}$$

Implement it with ReLU as the default activation. The checks verify the output shape, that ReLU never outputs negatives, and that the identity activation recovers the raw affine map.

In [ ]:
def relu(z):
    return np.maximum(z, 0.0)


def dense(X, W, b, activation=relu):
    """One dense layer: activation(X @ W + b)."""
    X = np.asarray(X, dtype=float)

    # TODO(you): the affine map X @ W + b, passed through the activation
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
X = rng.standard_normal((4, 3))
W = rng.standard_normal((3, 5))
b = rng.standard_normal(5)

out = dense(X, W, b)
assert out.shape == (4, 5), "batch of 4, width 5 -> output (4, 5)"
assert np.all(out >= 0), "ReLU output is never negative"

ident = dense(X, W, b, activation=lambda z: z)
assert np.allclose(ident, X @ W + b), "identity activation returns the raw affine map"
assert np.allclose(relu(ident), out), "dense = activation(X @ W + b)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dense(X, W, b, activation=relu):
    X = np.asarray(X, dtype=float)
    return activation(X @ W + b)
```

</details>

### Exercise 2 — Counting parameters

Each layer from width $d_{in}$ to $d_{out}$ costs $d_{in} \times d_{out}$ weights plus $d_{out}$ biases. Walk consecutive pairs of the size list and add it up. The classic MNIST MLP $[784, 128, 10]$ should come out to **101,770** — and note from the last check how parameter cost shapes architecture choices.

In [ ]:
def count_params(sizes):
    """Total trainable parameters of an MLP with the given layer sizes."""
    total = 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        # TODO(you): weights (n_in * n_out) plus biases (n_out)
        total += ...
    return total

In [ ]:
# Checks — run me
assert count_params([2, 3, 1]) == 13, "(2*3 + 3) + (3*1 + 1) = 13"
assert count_params([784, 128, 10]) == 101770, "the classic MNIST MLP"
assert count_params([5, 7]) == 42, "a single layer: 5*7 + 7"

deep = count_params([10, 100, 100, 1])
wide = count_params([10, 250, 1])
assert deep > wide, "two 100-wide layers cost more than one 250-wide layer here"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def count_params(sizes):
    total = 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        total += n_in * n_out + n_out
    return total
```

</details>